# 01 — Framing and deterministic baseline

**Question:** Can an evidence-grounded rule-based system triage synthetic IBM HI-Small accounts without any LLM or access to laundering labels?

**Hypothesis:** fixed account-level structural heuristics produce reproducible, transaction-traceable alerts but cannot establish criminal conduct.

**No data leakage:** raw labels are separated at ingestion, and the analyst tools only read unlabelled Parquet.

## A. Setup and full-dataset ingestion
Run this notebook with Jupyter started from the project root. Download the CSV first using the README command; the code intentionally does not invent data if absent.

In [1]:
from pathlib import Path
import json
import pandas as pd
from src.ingest import ingest
from src.prepare_data import prepare_subset
from src.evaluation import build_questions

raw = Path("data/raw/HI-Small_Trans.csv")
assert raw.exists(), "Download HI-Small_Trans.csv into data/raw first"
print("Raw CSV bytes:", raw.stat().st_size)

ModuleNotFoundError: No module named 'src'

In [ ]:
stats = ingest(raw, Path("data/processed"), Path("data/evaluation"))
print(json.dumps(stats, indent=2))

## B. Verify class distribution and source schema
These counts are measured from the CSV, not copied from a web description.

In [ ]:
import duckdb
con = duckdb.connect()
counts = con.execute("SELECT is_laundering, COUNT(*) n FROM read_parquet('data/evaluation/labels.parquet') GROUP BY 1 ORDER BY 1").df()
display(counts)
print("Positive rate:", stats["laundering_rate"])
print("Source SHA256:", stats["csv_sha256"])
display(con.execute("DESCRIBE SELECT * FROM read_parquet('data/processed/transactions.parquet')").df())

## C. Prepare complete ego-subnetwork subset
Positive-labelled seeds are selected only offline and must never be available to the agent. The subset is deliberately enriched; its class prevalence does not estimate deployment prevalence.

In [ ]:
subset_manifest = prepare_subset(
    Path("data/processed/transactions.parquet"),
    Path("data/evaluation/labels.parquet"),
    Path("data/processed/investigation_subset.parquet"),
    Path("data/evaluation"),
    n_positive=8, n_control=8, seed=42
)
print("Number of cases:", subset_manifest["n_cases"])
print("Transactions in local investigation subset:", subset_manifest["n_transactions"])
display(pd.DataFrame(subset_manifest["cases"]).head())

## D. Freeze evaluation tasks BEFORE building the LLM
`gold_answer` fields are placeholders until independently annotated from transactions or FATF pages.

In [ ]:
questions = build_questions(Path("data/processed/subset_manifest.json"), Path("data/evaluation/questions.jsonl"))
display(pd.DataFrame(questions)[["id", "category", "split", "question", "annotation_status"]])

## E. Run deterministic read-only baseline

In [ ]:
from src.functions import FinancialTools
from src.baseline import investigate
case_id = subset_manifest["cases"][0]["account_id"]
with FinancialTools(Path("data/processed/investigation_subset.parquet")) as tools:
    report = investigate(case_id, tools)
print(json.dumps(report, indent=2, default=str))

## F. Record observations, not fabricated performance
Fill in after running:

- Observed full CSV row count: **TBD**
- Observed full positive labels/rate: **TBD**
- Cases rejected for large ego graph: **TBD**
- Baseline pattern types and supporting transaction IDs: **TBD**
- Limitations: synthetic generator, uncalibrated thresholds, selection bias, no beneficial-owner information.

**Day-1 exit criterion:** show one real IBM account’s reported pattern, its raw numeric facts and its supporting synthetic transaction IDs.